# Agent Loop from First Principles

Built from scratch using **Gemini 2.5 Flash** — no frameworks, no magic.

## What is an Agent Loop?

An agent loop is the core of any AI agent:

```
┌─────────────────────────────────────────────┐
│                                             │
│   User message                              │
│       │                                     │
│       ▼                                     │
│   LLM decides: answer OR call a tool?       │
│       │                                     │
│       ├── tool_calls → run tool → back to LLM│
│       │                                     │
│       └── stop → return final answer        │
│                                             │
└─────────────────────────────────────────────┘
```

The LLM runs in a **loop** until it decides it has a final answer.
Each tool result is fed back in, so the LLM can chain multiple tool calls.

## Step 1 — Imports and setup

In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
# Create a Gemini client using the OpenAI-compatible endpoint
# The rest of the code is identical regardless of which model you use

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL = "gemini-2.5-flash-lite"

print(f"Client ready. Using model: {MODEL}")

Client ready. Using model: gemini-2.5-flash-lite


## Step 2 — Define the tools (Python functions)

We'll build a simple **calculator agent** with four arithmetic tools.
The agent will plan and execute multi-step calculations.

In [3]:
def add(a: float, b: float) -> float:
    """Add two numbers."""
    result = a + b
    print(f"  [tool] add({a}, {b}) = {result}")
    return result

def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    result = a - b
    print(f"  [tool] subtract({a}, {b}) = {result}")
    return result

def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    result = a * b
    print(f"  [tool] multiply({a}, {b}) = {result}")
    return result

def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        return "Error: division by zero"
    result = a / b
    print(f"  [tool] divide({a}, {b}) = {result}")
    return result

## Step 3 — Describe the tools in JSON (the schema)

The LLM doesn't see the Python functions — it only sees these JSON descriptions.
This is how it knows what tools exist and what arguments they take.

In [4]:
def make_arithmetic_tool(name, description):
    """Helper to build a tool schema for a two-number arithmetic function."""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"}
                },
                "required": ["a", "b"],
                "additionalProperties": False
            }
        }
    }

tools = [
    make_arithmetic_tool("add",      "Add two numbers together"),
    make_arithmetic_tool("subtract", "Subtract the second number from the first"),
    make_arithmetic_tool("multiply", "Multiply two numbers together"),
    make_arithmetic_tool("divide",   "Divide the first number by the second"),
]

print(f"{len(tools)} tools registered: {[t['function']['name'] for t in tools]}")

4 tools registered: ['add', 'subtract', 'multiply', 'divide']


## Step 4 — The tool dispatcher

When the LLM requests a tool call, we need to:
1. Find the matching Python function by name
2. Parse the arguments from the JSON the LLM produced
3. Call the function and capture the result
4. Return it in the format the LLM expects

In [5]:
def handle_tool_calls(tool_calls):
    """Execute all requested tool calls and return results as tool messages."""
    results = []
    for tool_call in tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        # Look up the Python function by name — no big if/elif needed!
        fn = globals().get(name)
        result = fn(**args) if fn else f"Unknown tool: {name}"

        # Package the result back for the LLM in the required format
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

## Step 5 — The Agent Loop

This is the heart of the agent. It's surprisingly simple:

- Call the LLM
- If it wants to call tools → run them, add results to the conversation, loop again
- If it's done (`stop`) → return the final answer

That's it. The loop is what makes it an *agent*.

In [6]:
def agent(user_message: str, system: str = None) -> str:
    """Run the agent loop until the LLM produces a final answer."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_message})

    iteration = 0
    while True:
        iteration += 1
        print(f"\n--- Loop iteration {iteration} ---")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        choice = response.choices[0]
        finish_reason = choice.finish_reason
        print(f"  finish_reason: {finish_reason}")

        if finish_reason == "tool_calls":
            # LLM wants to call tools — execute them and loop
            assistant_message = choice.message
            tool_results = handle_tool_calls(assistant_message.tool_calls)

            # Add the assistant's tool-call request AND the results to history
            messages.append(assistant_message)
            messages.extend(tool_results)

        else:
            # LLM is done — return the final answer
            final_answer = choice.message.content
            print(f"\n=== Final Answer ===")
            print(final_answer)
            return final_answer

## Step 6 — Run it!

Let's give the agent a multi-step problem that requires chaining several tool calls.

In [7]:
system_prompt = """You are a precise calculator assistant. 
Always use your tools to perform calculations — never calculate in your head.
Break multi-step problems into individual tool calls, one operation at a time.
Show your reasoning, then give the final answer."""

problem = """
A store sells 3 types of items:
- Apples: $1.20 each, bought 15
- Oranges: $0.85 each, bought 8  
- Bananas: $0.45 each, bought 22

There's a 10% discount on the total.
What is the final amount to pay?
"""

result = agent(problem, system=system_prompt)


--- Loop iteration 1 ---
  finish_reason: tool_calls
  [tool] multiply(15, 1.2) = 18.0

--- Loop iteration 2 ---
  finish_reason: tool_calls
  [tool] multiply(8, 0.85) = 6.8

--- Loop iteration 3 ---
  finish_reason: tool_calls
  [tool] multiply(22, 0.45) = 9.9

--- Loop iteration 4 ---
  finish_reason: tool_calls
  [tool] add(18, 6.8) = 24.8

--- Loop iteration 5 ---
  finish_reason: tool_calls
  [tool] add(24.8, 9.9) = 34.7

--- Loop iteration 6 ---
  finish_reason: tool_calls
  [tool] multiply(34.7, 0.1) = 3.4700000000000006

--- Loop iteration 7 ---
  finish_reason: tool_calls
  [tool] subtract(34.7, 3.47) = 31.230000000000004

--- Loop iteration 8 ---
  finish_reason: stop

=== Final Answer ===
The final amount to pay is $31.23.


## Step 7 — Try your own problem!

In [8]:
# Try your own multi-step problem here!

my_problem = """
If I invest $5000 at a 6% annual return, how much do I have after 3 years
using simple interest? Then subtract the original $5000 to find the profit.
"""

result = agent(my_problem, system=system_prompt)


--- Loop iteration 1 ---
  finish_reason: tool_calls
  [tool] multiply(5000, 0.06) = 300.0

--- Loop iteration 2 ---
  finish_reason: tool_calls
  [tool] multiply(300, 3) = 900

--- Loop iteration 3 ---
  finish_reason: tool_calls
  [tool] add(5000, 900) = 5900

--- Loop iteration 4 ---


RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash-lite\nPlease retry in 39.818424463s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-fl

## What just happened?

You built an agent from **5 core components**:

| Component | What it does |
|---|---|
| **Tools (Python functions)** | The actions the agent can take in the real world |
| **Tool schemas (JSON)** | How the LLM knows what tools exist and how to call them |
| **Tool dispatcher** | Bridges the LLM's JSON requests to actual Python function calls |
| **Message history** | The growing conversation that the LLM uses for context |
| **The loop** | Keeps running until `finish_reason != "tool_calls"` |

Every agentic framework (LangGraph, CrewAI, AutoGen) is essentially this loop, with more features on top.
Now you understand what they're all doing under the hood.